# EMS Data Profiling

Checks what the EPCR (Elite) and outcomes (ESO) data can actually answer before any analysis starts.

Run sections 0 through 3 first. They find the tables, flatten them, and pick the columns everything else uses.
If a column is picked wrong, set it by hand in `OVERRIDES` in section 0 and re-run from section 3.

The last section scores each question we want to answer against how complete the fields behind it are.

## 0. Config

- `YEAR_MIN` / `YEAR_MAX` — the window. Set to 2024 to 2026, because anything before 2024 was described as either inaccurate or simply not there. `YEAR_RELIABLE` marks 2025 as the first year anyone should quote externally.

In [ ]:
CATALOG = "prod"
EPCR_SCHEMA = "silver_elite_dwgmr"
EPCR_SCHEMA_ALT = "silver_elite_dwamgh"
ESO_KEYWORDS = ["eso", "outcome", "hospital", "hie", "discharge"]
ESO_TABLE = None

FACT = f"{CATALOG}.{EPCR_SCHEMA}.fact_incident"
DIMS = ["incident", "situation", "disposition", "agency", "patient", "payment", "scene", "cardiacarrest"]

YEAR_MIN = 2024
YEAR_MAX = 2026
YEAR_RELIABLE = 2025

OVERRIDES = {
    "incident_date": None,
    "incident_id": None,
    "patient_id": None,
    "payer": None,
    "primary_impression": None,
    "acuity": None,
    "disposition": None,
    "state": None,
    "county": None,
    "zip": None,
    "agency_name": None,
    "call_type": None,
}

BLANKS = ["", "null", "none", "n/a", "na", "unknown", "not recorded", "not applicable",
          "not reporting", "not known", "unable to complete"]

## 1. Helpers

### Helper functions

- `cols(table)` — column names, returns empty on error instead of raising, so a permissions problem shows as a skip rather than a crash.
- `tables(schema)` — table names, excluding temp views. Without the temp view filter, the views this notebook creates show up as if they were catalog tables.
- `search_columns(schema, keys)` — scans every column in every table for a keyword. Used to locate the payer and geography fields without knowing their names.
- `resolve(df, key, candidates)` — takes an ordered list of name fragments and returns the first column that matches, unless `OVERRIDES` already answers it.
- `is_blank(c)` — null, empty, or one of the `BLANKS` placeholders. This is the definition of missing used throughout.
- `completeness(df, columns)` — for each column, how many rows are not blank, and the percent.
- `top_values(df, column, n)` — value counts with percent of total.
- `overlap(left, lcol, right, rcol)` — distinct values on each side, how many appear in both, and match rate in each direction. Used for the EPCR to outcomes link.

In [ ]:
from pyspark.sql import functions as F

def cols(table):
    try:
        return [f.name for f in spark.table(table).schema.fields]
    except Exception:
        return []

def tables(schema):
    return [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect() if not r.isTemporary]

def search_columns(schema, keys):
    rows = []
    for t in tables(schema):
        for c in cols(f"{CATALOG}.{schema}.{t}"):
            if any(k.lower() in c.lower() for k in keys):
                rows.append((t, c))
    if not rows:
        return spark.createDataFrame([], "table string, column string")
    return spark.createDataFrame(rows, "table string, column string")

def resolve(df, key, keys, avoid=()):
    if OVERRIDES.get(key):
        return OVERRIDES[key]
    for k in keys:
        for c in df.columns:
            if k.lower() in c.lower() and not any(a.lower() in c.lower() for a in avoid):
                return c
    return None

def is_blank(c):
    return F.col(c).isNull() | F.lower(F.trim(F.col(c).cast("string"))).isin(BLANKS)

def completeness(df, columns):
    n = df.count()
    agg = df.agg(*[F.count(F.when(~is_blank(c), 1)).alias(c) for c in columns]).collect()[0].asDict()
    rows = [(k, int(v), n, round(100.0 * v / n, 1) if n else 0.0) for k, v in agg.items()]
    return spark.createDataFrame(rows, "field string, populated long, total long, pct_populated double").orderBy("pct_populated")

def top_values(df, column, n=25):
    total = df.count()
    return (df.groupBy(column).count()
              .withColumn("pct", F.round(100.0 * F.col("count") / total, 2))
              .orderBy(F.desc("count")).limit(n))

def overlap(left, lcol, right, rcol):
    l = left.select(F.col(lcol).cast("string").alias("k")).where("k is not null").distinct()
    r = right.select(F.col(rcol).cast("string").alias("k")).where("k is not null").distinct()
    ln, rn = l.count(), r.count()
    both = l.join(r, "k").count()
    return spark.createDataFrame(
        [(lcol, rcol, ln, rn, both,
          round(100.0 * both / ln, 1) if ln else 0.0,
          round(100.0 * both / rn, 1) if rn else 0.0)],
        "left_col string, right_col string, left_distinct long, right_distinct long, matched long, pct_of_left double, pct_of_right double")

## 2. What tables exist

Also checks whether `silver_elite_dwgmr` / `silver_elite_dwamgh` and the `silver_frn_qry_*` copies hold the same data, which came up as an open question.

### Which silver schemas exist

Lists every schema starting with `silver`. This is the list that came up in the meeting when the point was made that the old set was small and the new one has a lot of unfamiliar names in it.

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}").where("databaseName like 'silver%'"))

### Table and row counts

Counts rows and columns for every table in both Elite schemas.

This is what answers the open question from the meeting: whether `silver_elite_dwgmr` and `silver_frn_qry_elite_dwgmr_repl` hold the same data. The answer was reportedly yes, that the `frn_qry` copies are materialized from the same source. Matching row counts confirm it; a difference means one is stale and the analysis needs to pick a side.

`-1` in the rows column means the count failed, which in Unity Catalog usually means no read access rather than no table.

In [ ]:
inv = []
for s in [EPCR_SCHEMA, EPCR_SCHEMA_ALT]:
    for t in tables(s):
        full = f"{CATALOG}.{s}.{t}"
        try:
            inv.append((s, t, spark.table(full).count(), len(cols(full))))
        except Exception:
            inv.append((s, t, -1, -1))

display(spark.createDataFrame(inv, "schema string, table string, rows long, n_cols long").orderBy(F.desc("rows")))

### Where does payer live

Searches every column name in the EPCR schema for payment, payer, insurance, medicaid, or billing.

Payer is the field the whole Medicaid question rests on, and it is not obvious which table holds it. This finds every candidate so the right one can be picked deliberately.

In [ ]:
display(search_columns(EPCR_SCHEMA, ["payment", "payer", "insur", "medicaid", "billing"]))

### Where does geography live

Same search for ZIP, county, state, FIPS, census, latitude, longitude.

This determines the finest level any public data can be joined at later.

In [ ]:
display(search_columns(EPCR_SCHEMA, ["zip", "county", "state", "fips", "census", "lat", "lon"]))

## 3. Flatten the table

Joins `fact_incident` to its dimension tables so one row is one incident with all fields attached. Uses the `Dim_X_FK` to `Dim_X_PK` naming. Dimension columns get the dimension name in front so they do not collide.

### Find the join keys

`fact_incident` holds foreign keys named `Dim_Something_FK`. Each dimension holds a matching `Dim_Something_PK`.

This cell builds two lookups: foreign key by dimension name, and actual table name by dimension name. Both keys have underscores stripped, because the naming is not consistent between the two sides. The fact has `Dim_CardiacArrest_FK` while the table is `dim_cardiac_arrest` with a `Dim_Cardiac_Arrest_PK`. Matching on letters only handles that.

In [ ]:
fk_map = {}
for c in cols(FACT):
    if c.lower().endswith("_fk"):
        fk_map[c[:-3].replace("_", "").lower()] = c

dim_tables = {t.replace("_", "").lower(): t for t in tables(EPCR_SCHEMA)}

print(fk_map)
print(sorted(dim_tables))

### Check for column names SQL cannot read

Some Elite columns contain characters that are not valid in a bare SQL identifier, including question marks. Those columns caused a parse error on the first run.

This prints them so it is clear what is in the data. The flatten step below quotes every name with backticks, so nothing here blocks the run. It is worth reading anyway, since a question mark in a column name usually means the source system exported the survey question as the header.

In [ ]:
import re

odd = []
for d in DIMS + ["incident"]:
    for c in cols(f"{CATALOG}.{EPCR_SCHEMA}.dim_{d}"):
        if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c):
            odd.append((f"dim_{d}", c))
for c in cols(FACT):
    if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c):
        odd.append(("fact_incident", c))

print(len(odd))
for x in odd[:50]:
    print(x)

### Flatten the table

One row per incident with every dimension field attached.

For each dimension, the generated SQL looks like this:

```sql
SELECT
  fi.*,
  situation.`Situation_Provider_Primary_Impression` AS `situation_Situation_Provider_Primary_Impression`,
  payment.`Payment_Primary_Method_Of_Payment`      AS `payment_Payment_Primary_Method_Of_Payment`,
  ...
FROM prod.silver_elite_dwgmr.fact_incident fi
LEFT JOIN prod.silver_elite_dwgmr.dim_incident       incident      ON fi.`Dim_Incident_FK`      = incident.`Dim_Incident_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_situation      situation     ON fi.`Dim_Situation_FK`     = situation.`Dim_Situation_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_disposition    disposition   ON fi.`Dim_Disposition_FK`   = disposition.`Dim_Disposition_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_agency         agency        ON fi.`Dim_Agency_FK`        = agency.`Dim_Agency_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_patient        patient       ON fi.`Dim_Patient_FK`       = patient.`Dim_Patient_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_payment        payment       ON fi.`Dim_Payment_FK`       = payment.`Dim_Payment_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_scene          scene         ON fi.`Dim_Scene_FK`         = scene.`Dim_Scene_PK`
LEFT JOIN prod.silver_elite_dwgmr.dim_cardiacarrest  cardiacarrest ON fi.`Dim_CardiacArrest_FK` = cardiacarrest.`Dim_Cardiac_Arrest_PK`
```

Three things about how it is built:

- The primary key is read off each dimension table rather than derived from the foreign key name, because the two do not always agree on underscores.
- Every identifier is in backticks, which handles the odd column names found above.
- Dimension columns are prefixed with the dimension name, so `situation` and `disposition` can both have a `Created_On` without colliding.

All joins are LEFT, so an incident with no payment row still appears with nulls. That is deliberate. A missing dimension row is a finding, and an inner join would hide it.

The cell displays a plan table first: one row per dimension with the table, the keys used, and joined or skipped. Read that before moving on. A silent skip means fields are missing downstream with no error to warn you.

One caution on the row count. The join is only one-to-one if each `_PK` is unique in its dimension. If any dimension has duplicate keys, rows fan out and every count in the notebook is inflated. Compare `spark.table(FACT).count()` against `flat.count()` to rule that out.

In [ ]:
import re

def clean(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name)

def norm(name):
    return name.replace("_", "").lower()

sel = ["fi.*"]
joins = []
plan = []
for d in DIMS:
    key = norm("dim" + d)
    tbl = dim_tables.get(key)
    fk = fk_map.get(key)
    if not tbl or not fk:
        plan.append((d, tbl or "no table", fk or "no fk", "skipped"))
        continue
    full = f"{CATALOG}.{EPCR_SCHEMA}.{tbl}"
    c_all = cols(full)
    pks = [c for c in c_all if c.lower().endswith("_pk")]
    pk = next((c for c in pks if norm(c) == norm(fk)[:-2] + "pk"), pks[0] if pks else None)
    if not pk:
        plan.append((d, tbl, fk, "no pk found"))
        continue
    joins.append(f"LEFT JOIN {full} {d} ON fi.`{fk}` = {d}.`{pk}`")
    for c in c_all:
        if not c.lower().endswith("_pk"):
            sel.append(f"{d}.`{c}` AS `{d}_{clean(c)}`")
    plan.append((d, tbl, f"{fk} = {pk}", "joined"))

display(spark.createDataFrame(plan, "dim string, table string, keys string, status string"))

sql = f"SELECT {', '.join(sel)} FROM {FACT} fi " + " ".join(joins)
spark.sql(sql).createOrReplaceTempView("flat")
flat = spark.table("flat")
print(len(flat.columns), "columns")

### Pick the columns that matter

The flattened table has well over a thousand columns. This narrows it to the eleven the analysis needs.

Each entry gives an ordered list of name fragments, and the first match wins. `incident_date` tries `incident_Incident_Date` first, then `Incident_Date`, then `Unit_Notified`, and so on down to a bare `_Date`.

The displayed table is the one to audit before trusting anything downstream. If `payer` resolved to a billing type rather than the method of payment, or `patient_id` resolved to something that is not stable across visits, every number after this point inherits that mistake. Fix in `OVERRIDES` and re-run from section 3.

In [ ]:
FIELDS = {
    "incident_date":      resolve(flat, "incident_date", ["incident_Incident_Date", "Incident_Date", "Unit_Notified", "Dispatch_Date", "_Date"]),
    "incident_id":        resolve(flat, "incident_id", ["Incident_Transaction_GUID", "Incident_Number", "Response_Number", "PCR"]),
    "patient_id":         resolve(flat, "patient_id", ["patient_Patient_ID", "Patient_Key", "MRN", "Person_ID"]),
    "payer":              resolve(flat, "payer", ["payment_Primary_Method", "Payment_Method", "Payer", "Insurance_Company", "Billing_Type"]),
    "primary_impression": resolve(flat, "primary_impression", ["Primary_Impression", "Provider_Primary_Impression", "Impression"]),
    "acuity":             resolve(flat, "acuity", ["Acuity", "Severity", "Level_Of_Care", "Priority"]),
    "disposition":        resolve(flat, "disposition", ["Incident_Patient_Disposition", "Patient_Disposition", "disposition_Disposition"]),
    "state":              resolve(flat, "state", ["Scene_State", "Incident_State", "_State"]),
    "county":             resolve(flat, "county", ["Scene_County", "County"]),
    "zip":                resolve(flat, "zip", ["Scene_Zip", "Zip", "Postal"]),
    "agency_name":        resolve(flat, "agency_name", ["agency_Agency_Name", "Agency_Name", "agency_Name"]),
}

display(spark.createDataFrame([(k, v or "NOT FOUND") for k, v in FIELDS.items()], "field string, resolved_column string"))

### Add date parts

Pulls year, month, hour, and day of week off the incident timestamp so the volume and timing sections can group on them.

The counts by year printed here are the first sanity check. A year that looks far too small is usually partial coverage rather than a real volume change.

In [ ]:
DATE = FIELDS["incident_date"]
base = (flat.withColumn("yr", F.year(F.col(DATE).cast("timestamp")))
             .withColumn("mo", F.date_format(F.col(DATE).cast("timestamp"), "yyyy-MM"))
             .withColumn("hr", F.hour(F.col(DATE).cast("timestamp")))
             .withColumn("dow", F.date_format(F.col(DATE).cast("timestamp"), "E")))
base.createOrReplaceTempView("base")
display(base.groupBy("yr").count().orderBy("yr"))

## 4. Volume

Look for partial years and agencies dropping in or out. Both throw off year-over-year numbers.

### Volume by month

Monthly counts across the whole window.

Look for a ramp at the start and a drop at the end. Both are coverage artifacts, not demand. The meetings framed this work as understanding what we have before drawing conclusions, and this is where a partial year would quietly turn into a false trend.

In [ ]:
scope = base.where((F.col("yr") >= YEAR_MIN) & (F.col("yr") <= YEAR_MAX))
scope.createOrReplaceTempView("scope")
display(scope.groupBy("mo").count().orderBy("mo"))

### Volume by agency and year

One of the things Hillary was profiling was counts by distinct agency.

Read it for agencies that appear or disappear between years. An agency onboarding mid-period will look like growth, and one dropping out will look like decline, when neither is about patient demand.

In [ ]:
display(scope.groupBy("yr", FIELDS["agency_name"]).count().orderBy("yr", F.desc("count")))

### Volume by state

Shows the geographic footprint of the data, which sets the boundary on any state-level Medicaid comparison. Medicaid programs vary by state, so a national number built on a handful of states needs that said out loud.

In [ ]:
if FIELDS["state"]:
    display(scope.groupBy(FIELDS["state"]).count().orderBy(F.desc("count")))

## 5. Completeness

Blank counts nulls and NEMSIS placeholders like "Not Recorded" and "Not Applicable". Earlier profiling found about 50% patient coverage but only about 10% of fields filled in, so the placeholders matter.

### Completeness of the key fields

For each of the eleven resolved fields, how many of the in-scope incidents actually have a value.

This is the direct test of what was described in the meeting: a table where roughly half the patients are present but only around ten percent of the information is filled in. Blank here includes the NEMSIS placeholders, so this measures usable data rather than non-null data.

In [ ]:
KEY_FIELDS = [v for v in FIELDS.values() if v]
display(completeness(scope, KEY_FIELDS))

### Completeness of the clinical fields

Widens the same check to every column whose name mentions impression, symptom, complaint, vitals, medication, procedure, history, disposition, destination, level of care, or cardiac arrest. Capped at sixty columns.

This is what tells us whether clinical analysis is possible at all, or whether the fields exist but sit empty.

In [ ]:
CLINICAL = [c for c in scope.columns if any(k in c.lower() for k in
            ["impression", "symptom", "complaint", "vital", "medication", "procedure",
             "history", "disposition", "destination", "level_of_care", "cardiacarrest"])][:60]
display(completeness(scope, CLINICAL))

### Completeness by year

The same percentages split by year.

A field that is well populated in older years and empty in recent ones usually means a workflow or a vendor feed changed. That matters more than the overall average, because the recent years are the ones anyone will want to report on.

In [ ]:
n_by_yr = {r.yr: r["count"] for r in scope.groupBy("yr").count().collect()}
rows = []
for c in KEY_FIELDS:
    for r in scope.groupBy("yr").agg(F.count(F.when(~is_blank(c), 1)).alias("p")).collect():
        rows.append((c, r.yr, round(100.0 * r.p / n_by_yr[r.yr], 1)))
display(spark.createDataFrame(rows, "field string, yr int, pct_populated double").orderBy("field", "yr"))

## 6. Payer mix

Read the raw values before trusting the Medicaid flag. Managed care plan names often sit in the same column and will not match on the word medicaid.

### Raw payer values

Every distinct value in the payer field with counts, before any grouping.

Read this before the grouped version below. Two things commonly go wrong here: managed care organizations appear under their plan name rather than the word Medicaid, and a large generic bucket such as "Insurance" hides the payer entirely. Either one makes the Medicaid count a floor rather than an estimate, and that has to be stated whenever the number is used.

In [ ]:
PAYER = FIELDS["payer"]
display(top_values(scope, PAYER, 50))

### Group into payer buckets

Sorts the raw values into five groups. In SQL the logic is:

```sql
CASE
  WHEN lower(payer) RLIKE 'medicaid|chip|title xix|managed care' THEN 'Medicaid'
  WHEN lower(payer) RLIKE 'medicare'                             THEN 'Medicare'
  WHEN lower(payer) RLIKE 'self|patient pay|uninsur|no insur'    THEN 'Self Pay'
  WHEN payer IS NULL OR lower(trim(payer)) IN (...blanks...)     THEN 'Unknown'
  ELSE 'Other/Commercial'
END
```

Order matters: the first matching branch wins, so a value containing both "medicaid" and "managed care" lands in Medicaid.

The result by year is the starting point for the Medicaid share of EMS volume. Watch the size of Unknown. If it is large, or growing in the most recent year, the Medicaid share is understated by an unknown amount rather than measured.

In [ ]:
flagged = scope.withColumn("payer_group",
    F.when(F.lower(F.col(PAYER)).rlike("medicaid|chip|title xix|managed care"), "Medicaid")
     .when(F.lower(F.col(PAYER)).rlike("medicare"), "Medicare")
     .when(F.lower(F.col(PAYER)).rlike("self|patient pay|uninsur|no insur"), "Self Pay")
     .when(is_blank(PAYER), "Unknown")
     .otherwise("Other/Commercial"))
flagged.createOrReplaceTempView("flagged")
display(flagged.groupBy("yr", "payer_group").count().orderBy("yr", F.desc("count")))

### Payer mix by state

Cross-tab of state against payer group.

Useful for spotting a state where the payer field is coded differently, which shows up as one state having a wildly different mix from the rest.

In [ ]:
if FIELDS["state"]:
    display(flagged.groupBy(FIELDS["state"]).pivot("payer_group").count())

### What is actually inside the Insurance bucket

This is the open question from the call: Medicaid shows on the payer field about 4.4% of the time, while a generic "Insurance" value takes 38%. If Medicaid patients are sitting inside that bucket, every Medicaid number we produce is understated.

This cell narrows to just the Insurance rows and measures how complete every other column in the payment dimension is for them. A secondary method, plan name, or company field that is well populated on those rows is where the real payer is hiding.

In [ ]:
INS = scope.where(F.lower(F.trim(F.col(PAYER))) == "insurance")
pay_cols = [c for c in scope.columns if c.lower().startswith("payment_")][:40]
print(INS.count(), len(pay_cols))
insurance_bucket = completeness(INS, pay_cols)
display(insurance_bucket)

### Read the payer detail on those rows

Takes the payment columns most likely to name a plan or company and shows their top values, restricted to the Insurance rows.

If Medicaid managed care plan names turn up here, the fix is to widen the Medicaid rule in the grouping cell above to match those names, then re-run. That single change would move the Medicaid share materially.

In [ ]:
ins_detail = [c for c in scope.columns if c.lower().startswith("payment_")
              and any(k in c.lower() for k in ["company", "name", "secondary", "plan", "group", "type", "id"])][:6]
print(ins_detail)

for c in ins_detail:
    display(top_values(INS, c, 20))

### Is this a 911 call or a transfer between facilities

Raised on the call and worth settling early. Interfacility transfers are currently mixed into the reported numbers.

The two uses pull in opposite directions. For a payer or customer view, transfers are legitimate volume and belong in. For the question actually being chased, whether a Medicaid patient arriving through a 911 dispatch can be identified and routed differently, a scheduled transfer between two hospitals is not that patient and does not belong in the denominator.

This cell lists the columns that could carry the distinction so one can be chosen deliberately rather than by default.

In [ ]:
CALL_CANDIDATES = [c for c in scope.columns if any(k in c.lower() for k in
                   ["service_requested", "type_of_service", "response_type", "interfacility",
                    "transfer", "call_type", "incident_type", "scene_type", "complaint_reported"])][:40]
print(CALL_CANDIDATES)

### Split the volume by call origin

Classifies each incident as a scene response, an interfacility transfer, or unclear, and adds it to the working table as `call_origin`.

Read the unclear share first. If it is large, the field picked is not the right one and another candidate from the list above should go into `OVERRIDES` as `call_type`.

Because this is added to `flagged`, the Medicaid subset built in the next section inherits it. That means the transfer question can be answered either way without re-running everything: keep them for the payer view, filter to scene responses for the 911 routing view.

In [ ]:
CALLTYPE = OVERRIDES.get("call_type") or (CALL_CANDIDATES[0] if CALL_CANDIDATES else None)
print(CALLTYPE)

call_origin_mix = None
if CALLTYPE:
    display(top_values(scope, CALLTYPE, 30))
    flagged = flagged.withColumn("call_origin",
        F.when(F.lower(F.col(CALLTYPE)).rlike("interfacility|inter-facility|ift|transfer"), "Interfacility")
         .when(F.lower(F.col(CALLTYPE)).rlike("911|9-1-1|emergency|scene|dispatch"), "911 Scene")
         .otherwise("Other/Unclear"))
    flagged.createOrReplaceTempView("flagged")
    call_origin_mix = flagged.groupBy("yr", "payer_group", "call_origin").count().orderBy("yr", F.desc("count"))
    display(call_origin_mix)

### Medicaid mix by county

County is the level payer conversations happen at, so this is the cut that supports a county-by-county view.

It is also the shape of the public data join. ACS, HPSA, and CDC PLACES are all available at county level, so this is the table those would attach to.

In [ ]:
payer_by_county = None
if FIELDS["county"]:
    payer_by_county = (flagged.groupBy(FIELDS["state"], FIELDS["county"], "payer_group").count()
                              .orderBy(F.desc("count")).limit(200))
    display(payer_by_county)

## 7. How often patients come back

Groups patients into 1, 2-4, 5-11, and 12+ encounters. First check whether the patient ID stays the same across incidents. If incidents per patient comes back near 1.0, the ID is created fresh each call and none of this can be built from this table.

### Is the patient ID stable

Run this before reading anything below it.

It counts Medicaid incidents, distinct patient IDs, and the ratio. If incidents per patient comes back near 1.0, the identifier is generated per encounter rather than per person. In that case the tiers and the repeat interval below are measuring nothing, and repeat utilization cannot be built from this table at all.

That would be a significant finding on its own, since the utilization pyramid and the recurrent demand model are both named as high-value products in the Medicaid write-up.

In [ ]:
PID = FIELDS["patient_id"]
med = flagged.where("payer_group = 'Medicaid'")
med.createOrReplaceTempView("med")
display(med.agg(F.count("*").alias("incidents"),
                F.countDistinct(PID).alias("distinct_patients"),
                F.round(F.count("*") / F.countDistinct(PID), 2).alias("incidents_per_patient")))

### Encounter tiers

Groups Medicaid patients into 1, 2 to 4, 5 to 11, and 12 or more encounters, then reports both the share of patients and the share of encounters in each tier.

The point of the tiers is the gap between those two percentages. National work on this shows a small group of high utilizers accounting for a disproportionate share of encounters, and the two columns side by side are what make that visible.

This output is only meaningful if the previous cell showed a patient ID that persists across visits.

In [ ]:
per_pt = med.where(F.col(PID).isNotNull()).groupBy(PID).agg(F.count("*").alias("encounters"))
pyramid = (per_pt.withColumn("tier", F.when(F.col("encounters") == 1, "1")
                                      .when(F.col("encounters") <= 4, "2-4")
                                      .when(F.col("encounters") <= 11, "5-11")
                                      .otherwise("12+"))
                 .groupBy("tier").agg(F.count("*").alias("patients"), F.sum("encounters").alias("encounters")))
total_pt = per_pt.count()
total_enc = per_pt.agg(F.sum("encounters")).collect()[0][0]
pyramid_out = (pyramid.withColumn("pct_patients", F.round(100.0 * F.col("patients") / total_pt, 2))
                      .withColumn("pct_encounters", F.round(100.0 * F.col("encounters") / total_enc, 2)))
display(pyramid_out)

### Time between encounters

For each Medicaid patient, sorts their incidents by date and measures the gap to the next one:

```sql
LEAD(incident_date) OVER (PARTITION BY patient_id ORDER BY incident_date)
```

Then reports the median gap and the share of repeat encounters falling within 7 and within 30 days.

These are the two windows named in the recurrent demand model. Same dependency as above: without a stable patient ID this is not measuring return visits.

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy(PID).orderBy(F.col(DATE).cast("timestamp"))
gaps = (med.where(F.col(PID).isNotNull())
           .withColumn("next_dt", F.lead(F.col(DATE).cast("timestamp")).over(w))
           .withColumn("days_to_next", F.datediff("next_dt", F.col(DATE).cast("timestamp")))
           .where("days_to_next is not null"))
display(gaps.agg(F.count("*").alias("pairs"),
                 F.expr("percentile_approx(days_to_next, 0.5)").alias("median_days"),
                 F.round(100.0 * F.avg(F.when(F.col("days_to_next") <= 7, 1).otherwise(0)), 1).alias("pct_within_7d"),
                 F.round(100.0 * F.avg(F.when(F.col("days_to_next") <= 30, 1).otherwise(0)), 1).alias("pct_within_30d")))

## 8. Clinical and timing

### Top primary impressions

The most common primary impressions among Medicaid incidents.

This was one of the specific things being profiled. It is also the input to any avoidable-episode work, since categories such as abdominal pain, chest and throat pain, and alcohol-related presentations are the ones that analysis flagged. Treating everything here as low acuity would be wrong, which is exactly why it is worth looking at the actual distribution first.

In [ ]:
display(top_values(med, FIELDS["primary_impression"], 30))

### Secondary impression and comorbidities

The primary impression alone is one line of a clinical picture. The secondary impression and the comorbidity fields are where the chronic and psychiatric burden shows up, and the two together are what make a case that this is a complex population rather than low acuity misuse.

This lists the secondary impression columns and profiles the most common values for Medicaid.

In [ ]:
SEC = [c for c in scope.columns if "impression" in c.lower() and c != FIELDS["primary_impression"]]
print(SEC)

secondary_impressions = None
if SEC:
    secondary_impressions = top_values(med, SEC[0], 30)
    display(secondary_impressions)

### Behavioral health share of Medicaid EMS

Flags incidents whose primary impression mentions behavioral, psychiatric, substance, overdose, alcohol, suicide, or a named psychiatric condition.

This is the EPCR-side counterpart to the nurse navigation note screening. That work runs on free-text nurse notes; this runs on coded impressions, so the two are independent measurements of the same underlying population. If they land in a similar range, that is worth a lot more than either number alone.

Also note the caveat from the call: an acronym match inside a longer word produces false hits. This matches on whole words only for the short ones.

In [ ]:
BH = "behavioral|psychiatric|anxiety|depress|bipolar|schizo|suicide|substance|alcohol|overdose"
IMP = FIELDS["primary_impression"]

bh = med.withColumn("behavioral", F.lower(F.col(IMP)).rlike(BH))
behavioral_health = bh.groupBy("yr").agg(F.count("*").alias("medicaid_incidents"),
                                         F.sum(F.col("behavioral").cast("int")).alias("behavioral"),
                                         F.round(100.0 * F.avg(F.col("behavioral").cast("int")), 1).alias("pct")).orderBy("yr")
display(behavioral_health)
display(top_values(bh.where("behavioral"), IMP, 25))

### Demand by hour and day

Counts by day of week and hour of day.

The claim worth testing is that weekday afternoons carry the heaviest volume while overnight has the worst response times, which would mean staffing models built around a morning peak are aimed at the wrong hours. Note that days sort alphabetically, so read the labels rather than the row order.

In [ ]:
display(med.groupBy("dow", "hr").count().orderBy("dow", "hr"))

### Disposition mix

What happened to the patient: transported, treated and released, refused, and so on.

This separates transports from non-transports, which is the difference between a billable event and unpaid work. Check the completeness number for this field before reading too much into the distribution.

In [ ]:
if FIELDS["disposition"]:
    display(top_values(med, FIELDS["disposition"], 30))

### Acuity by payer group

Cross-tab of acuity against payer group.

Relevant to the claim that Medicaid EMS demand is not simply low acuity misuse. If the acuity distribution for Medicaid resembles the other payer groups, that claim has support in our own data rather than only in the national literature.

In [ ]:
if FIELDS["acuity"]:
    display(flagged.groupBy(FIELDS["acuity"], "payer_group").count().orderBy(F.desc("count")))

### What history fields exist

Lists any column mentioning history or comorbidity.

The Medicaid write-up leans on the comorbidity picture, hypertension and diabetes alongside a heavy mental health burden. This prints whether the fields behind that exist here before anyone plans work that depends on them.

In [ ]:
hist = [c for c in scope.columns if "history" in c.lower() or "comorb" in c.lower()]
print(hist)

## 9. ESO outcomes

`silver_frn_qry_eso_ems` turned out to be an empty schema. The first cell searches every catalog for tables matching `ESO_KEYWORDS`, the second tests each for readability and row count, and the third picks the largest readable one. Set `ESO_TABLE` in section 0 to a full three part name to pin a specific table.

If nothing is found or nothing is readable, the rest of this section skips itself and the run continues. An empty result here is itself the finding: either the outcomes data is not landed in this workspace, or the grants are not in place. Unity Catalog reports objects you cannot read as missing, so the two look identical from here and need someone with grant visibility to tell apart.

Two things to find out: how many transported patients have an outcome record at all, and whether the ones that match are a skewed group. Hospitals were reportedly not sending low acuity records.

### Find the outcomes tables

The schema we were pointed to came back empty, so this searches the metadata catalog for any table whose schema or name matches `ESO_KEYWORDS`.

It queries `system.information_schema.tables` first, which spans every catalog, and falls back to the prod one if that is not granted. Metadata search works even where the data itself is not readable, so this returns results either way.

In [ ]:
where = " OR ".join([f"lower(table_schema) LIKE '%{k}%' OR lower(table_name) LIKE '%{k}%'" for k in ESO_KEYWORDS])
cand = None
for src in ["system.information_schema.tables", f"{CATALOG}.information_schema.tables"]:
    try:
        cand = spark.sql(f"SELECT table_catalog, table_schema, table_name, table_type FROM {src} WHERE {where} ORDER BY table_catalog, table_schema, table_name")
        cand.count()
        print(src)
        break
    except Exception as e:
        print(src, str(e)[:120])

eso_candidates = [f"{r.table_catalog}.{r.table_schema}.{r.table_name}" for r in cand.limit(200).collect()] if cand is not None else []
print(len(eso_candidates))
if cand is not None:
    display(cand)

### Test each candidate

Tries to count each candidate table and records the outcome.

The status column carries the real error rather than hiding it. If everything comes back not found, that is permissions rather than naming, since Unity Catalog reports objects you cannot read as missing. That distinction cannot be settled from here and needs someone with grant visibility.

In [ ]:
eso_inv = []
for full in eso_candidates:
    try:
        eso_inv.append((full, spark.table(full).count(), len(cols(full)), "ok"))
    except Exception as e:
        eso_inv.append((full, -1, -1, str(e)[:150]))

eso_inv.sort(key=lambda x: -x[1])
display(spark.createDataFrame(eso_inv, "table string, rows long, n_cols long, status string"))

### Pick a table

Takes the largest readable candidate, or whatever is pinned in `ESO_TABLE`.

If nothing is readable, `eso` stays empty, the rest of this section skips itself, and the run continues to the export. The other sections do not depend on outcomes data.

In [ ]:
readable = [r[0] for r in eso_inv if r[3] == "ok"]
print(readable)

if ESO_TABLE is None and readable:
    ESO_TABLE = readable[0]

eso = spark.table(ESO_TABLE) if ESO_TABLE else None
print(ESO_TABLE)

if eso is not None:
    print(eso.count(), len(eso.columns))
    display(spark.createDataFrame([(c,) for c in eso.columns], "column string"))

### Candidate join keys

Lists columns on each side that look like they could be the link: incident, response, PCR, run, record, GUID, event.

There is no documented key between the EPCR data and the outcomes feed, so this is the step where one gets chosen. Print both lists and compare them by eye before accepting the automatic pick.

In [ ]:
eso_keys = [c for c in eso.columns if any(k in c.lower() for k in ["incident", "response", "pcr", "run", "record", "guid", "event"])] if eso is not None else []
epcr_keys = [c for c in scope.columns if any(k in c.lower() for k in ["incident_number", "response_number", "transaction_guid", "pcr"])]
print(eso_keys)
print(epcr_keys)

### Test the key

Counts distinct values on each side, how many appear in both, and the match rate in each direction.

Read the direction. A key that matches a high share of outcomes records but a low share of EPCR records means the outcomes feed covers only some agencies or some hospitals. Both low means the key is wrong.

Zero matches is almost always a wrong key rather than missing data. A real coverage gap gives a small number, not none.

In [ ]:
ESO_KEY = eso_keys[0] if eso_keys else None
EPCR_KEY = FIELDS["incident_id"]

if ESO_KEY:
    display(overlap(scope, EPCR_KEY, eso, ESO_KEY))

### Completeness of the outcome fields

For columns mentioning outcome, diagnosis, disposition, discharge, admission, mortality, ICD, ED, or hospital, how many rows have a value.

This is the second half of the problem described in the meeting. Even where a record exists, the fields inside it were reported as largely empty, so record coverage and field completeness have to be measured separately.

In [ ]:
if eso is not None:
    OUTCOME_COLS = [c for c in eso.columns if any(k in c.lower() for k in
                    ["outcome", "diagnosis", "disposition", "discharge", "admit", "admission",
                     "mortality", "death", "icd", "ed_", "hospital"])][:40]
    display(completeness(eso, OUTCOME_COLS))

### Match rate by year

Left joins the distinct outcome keys onto the EPCR incidents and flags each incident as matched or not:

```sql
SELECT scope.*, (k.key IS NOT NULL) AS has_outcome
FROM scope
LEFT JOIN (SELECT DISTINCT eso_key AS key FROM eso) k
  ON CAST(scope.incident_id AS STRING) = k.key
```

The distinct is there so a patient with several outcome rows does not duplicate the EPCR row.

The result is the share of transports with an outcome record, by year. The figure mentioned in the meeting was around half the patients present for 2025, which gives a reference point for whether this looks right.

In [ ]:
matched = None
eso_match_yr = None

if ESO_KEY:
    matched = (scope.join(eso.select(F.col(ESO_KEY).cast("string").alias("_k")).distinct(),
                          F.col(EPCR_KEY).cast("string") == F.col("_k"), "left")
                    .withColumn("has_outcome", F.col("_k").isNotNull()))
    eso_match_yr = matched.groupBy("yr").agg(F.count("*").alias("epcr_records"),
                                             F.sum(F.col("has_outcome").cast("int")).alias("with_outcome"),
                                             F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy("yr")
    display(eso_match_yr)

### Match rate by acuity

The same match rate broken out by acuity.

This is the bias test. The reported problem was that hospitals were not sending low acuity records. If the match rate is high for serious cases and near zero for minor ones, then the matched set is not a sample of our patients, it is a sample of our sickest patients, and no outcome rate computed from it generalizes.

In [ ]:
if matched is not None and FIELDS["acuity"]:
    display(matched.groupBy(FIELDS["acuity"]).agg(F.count("*").alias("n"),
            F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy(F.desc("n")))

### Match rate by agency

The same match rate by agency, top 40 by volume.

Separates a data-sharing problem from a data-quality problem. If a handful of agencies match well and the rest match at zero, the outcomes feed is scoped to specific markets or specific hospital partners rather than being broadly incomplete.

In [ ]:
if matched is not None:
    display(matched.groupBy(FIELDS["agency_name"]).agg(F.count("*").alias("n"),
            F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy(F.desc("n")).limit(40))

## 10. Geography

Shows the smallest level we can join public data (ACS, HPSA, CDC PLACES) to. County FIPS or ZIP is the realistic target.

### Geography completeness

How often state, county, and ZIP are populated on Medicaid incidents.

Whichever of these is well populated sets the join level for public data. County is usually the realistic answer; ZIP is better if it is there.

In [ ]:
GEO = [v for v in [FIELDS["state"], FIELDS["county"], FIELDS["zip"]] if v]
display(completeness(med, GEO))

### Where the Medicaid volume is

Top state and county combinations by incident count.

Gives the concentration of Medicaid volume and the specific counties any public data pull would need to cover.

In [ ]:
if FIELDS["county"]:
    display(med.groupBy(FIELDS["state"], FIELDS["county"]).count().orderBy(F.desc("count")).limit(50))

## 11. What we can answer

Each question is scored by the weakest field it needs. Anything under 80% needs a fix or a caveat before it goes in front of leadership.

### What we can answer

Each question is scored by the least complete field it depends on. 80 or above is Ready, 40 to 80 needs a caveat, below 40 is blocked.

This list is the draft of what gets taken back to Noah and Rex: here is what the data can answer, you tell us which ones matter and what you would do with the answer. The point of scoring them is that the conversation starts from what is actually possible rather than from a general offer to analyze.

Read it with one limitation in mind: this measures whether fields are filled in, not whether they are correct or usable. A patient ID that is populated on every row still scores 100 even if it is regenerated each encounter and useless for tracking people. Likewise the outcome linkage row only checks that the EPCR key exists, not that anything matches it. The scorecard flags empty fields; it cannot flag fields that are full of the wrong thing.

In [ ]:
QUESTIONS = {
    "Medicaid share of EMS volume": [PAYER, DATE],
    "Medicaid mix by county": [FIELDS["county"], PAYER],
    "911 scene vs interfacility split": [PAYER, DATE],
    "Behavioral health share of Medicaid EMS": [FIELDS["primary_impression"], PAYER],
    "Utilization pyramid (1 / 2-4 / 5-11 / 12+)": [PID, PAYER, DATE],
    "7- and 30-day recurrent demand": [PID, DATE],
    "Clinical mix / potentially avoidable episodes": [FIELDS["primary_impression"], PAYER],
    "Time-of-day and day-of-week demand": [DATE],
    "Transport vs non-transport disposition": [FIELDS["disposition"], PAYER],
    "Acuity mix by payer": [FIELDS["acuity"], PAYER],
    "Geographic overlay with ACS / HPSA / PLACES": [FIELDS["county"], FIELDS["zip"], FIELDS["state"]],
    "ED outcome linkage": [EPCR_KEY],
}

needed = list({f for v in QUESTIONS.values() for f in v if f})
pct = {r.field: r.pct_populated for r in completeness(med, needed).collect()}
rows = []
for q, fs in QUESTIONS.items():
    fs = [f for f in fs if f]
    worst = min([pct.get(f, 0.0) for f in fs]) if fs else 0.0
    verdict = "Ready" if worst >= 80 else ("Caveat needed" if worst >= 40 else "Blocked")
    rows.append((q, ", ".join(fs), worst, verdict))

scorecard = spark.createDataFrame(rows, "question string, fields string, weakest_field_pct double, verdict string").orderBy(F.desc("weakest_field_pct"))
display(scorecard)

## 12. Save results

Writes one workbook to a `results` folder next to this notebook.

### Set up the output folder

Reads the notebook's own path and creates a `results` folder beside it. Prints the path so it can be confirmed before anything is written.

In [ ]:
import os
import pandas as pd

try:
    import openpyxl
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
RESULTS = os.path.join("/Workspace", os.path.dirname(nb_path).lstrip("/"), "results")
os.makedirs(RESULTS, exist_ok=True)
print(RESULTS)

### Write the workbook

One Excel file, one sheet per result. Each sheet is capped at 5,000 rows; volume by agency and the hour-by-day grid are the two that could hit that.

Sheets that depend on outcomes data are added only if that data was found, so the export works either way.

This is also the de-identified extract that was asked for. It carries counts and rates only, no patient names, so it can be shared without going through a data access request.

In [ ]:
sheets = {
    "table_inventory": spark.createDataFrame(inv, "schema string, table string, rows long, n_cols long"),
    "join_plan": spark.createDataFrame(plan, "dim string, table string, keys string, status string"),
    "resolved_fields": spark.createDataFrame([(k, v or "NOT FOUND") for k, v in FIELDS.items()], "field string, resolved_column string"),
    "volume_by_year": scope.groupBy("yr").count().orderBy("yr"),
    "volume_by_month": scope.groupBy("mo").count().orderBy("mo"),
    "volume_by_agency": scope.groupBy("yr", FIELDS["agency_name"]).count().orderBy("yr", F.desc("count")),
    "completeness": completeness(scope, KEY_FIELDS),
    "payer_values": top_values(scope, PAYER, 50),
    "payer_mix": flagged.groupBy("yr", "payer_group").count().orderBy("yr", F.desc("count")),
    "medicaid_pyramid": pyramid_out,
    "top_impressions": top_values(med, FIELDS["primary_impression"], 30),
    "hour_by_day": med.groupBy("dow", "hr").count().orderBy("dow", "hr"),
    "geo_completeness": completeness(med, GEO),
    "scorecard": scorecard,
}

for name, df in [("insurance_bucket", insurance_bucket),
                 ("call_origin_mix", call_origin_mix),
                 ("payer_by_county", payer_by_county),
                 ("secondary_impressions", secondary_impressions),
                 ("behavioral_health", behavioral_health),
                 ("eso_match_by_year", eso_match_yr)]:
    if df is not None:
        sheets[name] = df

path = os.path.join(RESULTS, "ems_data_profiling_results.xlsx")
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    for name, df in sheets.items():
        df.limit(5000).toPandas().to_excel(writer, sheet_name=name[:31], index=False)

print(path)

## 13. Open questions

- What is inside the generic Insurance value on the payer field? Section 6 narrows it down. Until that is answered every Medicaid figure here is a floor, not an estimate.
- Do we keep or exclude interfacility transfers? Different answer depending on whether the audience is a payer looking at total volume or the 911 routing question. Section 6 splits them so both are available.
- Does the patient ID stay the same across incidents or get created fresh each call? The first cell in section 7 answers it, and all the repeat use work depends on it.
- Does the payer field come from billing or from the crew at the scene? Changes how far the Medicaid flag can be trusted. Revenue cycle data is expected in about three weeks and should settle it.
- Is the ESO match gap missing data or a bad join key? Confirm before treating it as missing.
- Where do the nurse navigation calls and the EPCR records overlap? The link is patients who were transported, since those appear in both. That join is the basis for measuring whether a routing decision worked.
- Image Trend (`silver_elite_dwamgh`) access, and whether it needs to be added in for national coverage.

Cross-check worth keeping: a rough Genie query put Medicaid at roughly 230,000 patients last year, which is close to what this notebook produces for 2025. Two different routes to the same number is a reason to trust it.